## Python Code for Data Preprocessing of Nanomedicine Research Papers

Below is the Python code that performs the specified data preprocessing steps on nanomedicine research papers. The code includes:

1. Text Cleaning:
* Lowercasing
* Removing punctuation and numbers
* Stop words removal
* Lemmatization
2. Handling Special Terms:
* Creating a custom dictionary to preserve nanomedicine-specific terminology
3. De-duplication:
* Removing duplicate entries

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import string
import nltk
import json

# Download NLTK data files (only need to run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
# Function to remove duplicates
def remove_duplicates(df):
    # Remove duplicate rows based on the 'title' and 'abstract' columns
    df_dedup = df.drop_duplicates(subset=['title', 'abstract'])
    return df_dedup

In [ ]:
# Function to preprocess text
def preprocess_text(text, custom_terms=None):
    # Convert to lowercase
    text = text.lower()
    
    if custom_terms is not None:
        # Replace special nanomedicine terms with placeholders to preserve them
        for term in custom_terms:
            # Replace spaces in term with underscores
            term_underscore = term.replace(' ', '_')
            # Use regex to replace the term in text
            text = re.sub(r'\b' + re.escape(term.lower()) + r'\b', term_underscore, text)
    
    # Remove punctuation and numbers
    text = re.sub(r'[{}]'.format(string.punctuation), ' ', text)
    text = re.sub(r'\d+', '', text)
    
    # Tokenize the text
    tokens = text.split()
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    if custom_terms is not None:
        # Replace placeholders back to original terms with spaces
        tokens = [word.replace('_', ' ') if word in [t.replace(' ', '_') for t in custom_terms] else word for word in tokens]

    # Join tokens back into a string
    processed_text = ' '.join(tokens)
    
    return processed_text

In [ ]:
folder_path = 'papers'
json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]

if os.path.exists('processed_papers') == False:
    os.makedirs('processed_papers')

for file in json_files:
    with open(os.path.join(folder_path, file), 'r') as f:
        data = json.load(f)

        # Preprocess the abstract and title
        cleaned_abstract = preprocess_text(data['abstract'])

        # append title to abstract
        data['cleaned_text'] = data['title'] + ' ' + cleaned_abstract

        with open(os.path.join('processed_papers', file), 'w') as f:
            json.dump(data, f, indent=4)